# FinRL Hyperparameter Optimization with Walk-Forward Validation

**Purpose**: This notebook performs hyperparameter optimization (HPO) using walk-forward validation to find robust configurations for the PPO trading agent.

**Output**: Best hyperparameters to copy into the v2 trading notebook for deployment.

---

*Disclaimer: Nothing herein is financial advice, and NOT a recommendation to trade real money.*

<a target="_blank" href="https://colab.research.google.com/github/AI4Finance-Foundation/FinRL-Tutorials/blob/master/3-Practical/FinRL_PaperTrading_Demo.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Part 1: Install FinRL

In [ ]:
## install finrl library
#!pip install wrds
#!pip install swig
#!pip install -q condacolab
#import condacolab
#condacolab.install()
#!apt-get update -y -qq && apt-get install -y -qq cmake libopenmpi-dev python3-dev zlib1g-dev libgl1-mesa-glx swig
#!pip install git+https://github.com/AI4Finance-Foundation/FinRL.git

## Import related modules

In [25]:
from finrl.config_tickers import DOW_30_TICKER
from finrl.config import INDICATORS
from finrl.meta.env_stock_trading.env_stocktrading_np import StockTradingEnv
from finrl.meta.env_stock_trading.env_stock_papertrading import AlpacaPaperTrading
from finrl.meta.data_processor import DataProcessor
from finrl.plot import backtest_stats, backtest_plot, get_daily_return, get_baseline

import numpy as np
import pandas as pd

## PPO

In [27]:
import os
import time
import gymnasium as gym
import numpy as np
import numpy.random as rd
import torch
import torch.nn as nn
from copy import deepcopy
from torch import Tensor
from torch.distributions.normal import Normal


def check_cuda_kernel():
    if not torch.cuda.is_available():
        print("CUDA not available. Using CPU.")
        return torch.device("cpu")
    try:
        # Allocation alone doesn't trigger kernels — force a real compute op
        x = torch.randn(2, 2, device='cuda')
        _ = torch.relu(x @ x)           # matmul + relu = real kernel execution
        torch.cuda.synchronize()         # surface async errors immediately
        print(f"CUDA OK: {torch.cuda.get_device_name(0)} (compute {torch.cuda.get_device_capability()})")
        return torch.device("cuda")
    except RuntimeError as e:
        if "no kernel image" in str(e) or "CUDA error" in str(e):
            print(f"CUDA kernel incompatible with {torch.cuda.get_device_name(0)}.")
            print("Falling back to CPU.")
            return torch.device("cpu")
        raise

device = check_cuda_kernel()


class ActorPPO(nn.Module):
    def __init__(self, dims: [int], state_dim: int, action_dim: int):
        super().__init__()
        self.net = build_mlp(dims=[state_dim, *dims, action_dim])
        self.action_std_log = nn.Parameter(torch.zeros((1, action_dim)), requires_grad=True)  # trainable parameter

    def forward(self, state: Tensor) -> Tensor:
        return self.net(state).tanh()  # action.tanh()

    def get_action(self, state: Tensor) -> (Tensor, Tensor):  # for exploration
        action_avg = self.net(state)
        action_std = self.action_std_log.exp()

        dist = Normal(action_avg, action_std)
        action = dist.sample()
        logprob = dist.log_prob(action).sum(1)
        return action, logprob

    def get_logprob_entropy(self, state: Tensor, action: Tensor) -> (Tensor, Tensor):
        action_avg = self.net(state)
        action_std = self.action_std_log.exp()

        dist = Normal(action_avg, action_std)
        logprob = dist.log_prob(action).sum(1)
        entropy = dist.entropy().sum(1)
        return logprob, entropy

    @staticmethod
    def convert_action_for_env(action: Tensor) -> Tensor:
        return action.tanh()


class CriticPPO(nn.Module):
    def __init__(self, dims: [int], state_dim: int, _action_dim: int):
        super().__init__()
        self.net = build_mlp(dims=[state_dim, *dims, 1])

    def forward(self, state: Tensor) -> Tensor:
        return self.net(state)  # advantage value


def build_mlp(dims: [int]) -> nn.Sequential:  # MLP (MultiLayer Perceptron)
    net_list = []
    for i in range(len(dims) - 1):
        net_list.extend([nn.Linear(dims[i], dims[i + 1]), nn.ReLU()])
    del net_list[-1]  # remove the activation of output layer
    return nn.Sequential(*net_list)


class Config:
    def __init__(self, agent_class=None, env_class=None, env_args=None):
        self.env_class = env_class  # env = env_class(**env_args)
        self.env_args = env_args  # env = env_class(**env_args)

        if env_args is None:  # dummy env_args
            env_args = {'env_name': None, 'state_dim': None, 'action_dim': None, 'if_discrete': None}
        self.env_name = env_args['env_name']  # the name of environment. Be used to set 'cwd'.
        self.state_dim = env_args['state_dim']  # vector dimension (feature number) of state
        self.action_dim = env_args['action_dim']  # vector dimension (feature number) of action
        self.if_discrete = env_args['if_discrete']  # discrete or continuous action space

        self.agent_class = agent_class  # agent = agent_class(...)

        '''Arguments for reward shaping'''
        self.gamma = 0.99  # discount factor of future rewards
        self.reward_scale = 1.0  # an approximate target reward usually be closed to 256

        '''Arguments for training'''
        self.gpu_id = 0 if device.type == 'cuda' else -1
        self.net_dims = (64, 32)  # the middle layer dimension of MLP (MultiLayer Perceptron)
        self.learning_rate = 6e-5  # 2 ** -14 ~= 6e-5
        self.soft_update_tau = 5e-3  # 2 ** -8 ~= 5e-3
        self.batch_size = int(128)  # num of transitions sampled from replay buffer.
        self.horizon_len = int(2000)  # collect horizon_len step while exploring, then update network
        self.buffer_size = None  # ReplayBuffer size. Empty the ReplayBuffer for on-policy.
        self.repeat_times = 8.0  # repeatedly update network using ReplayBuffer to keep critic's loss small

        '''Arguments for evaluate'''
        self.cwd = None  # current working directory to save model. None means set automatically
        self.break_step = +np.inf  # break training if 'total_step > break_step'
        self.eval_times = int(32)  # number of times that get episodic cumulative return
        self.eval_per_step = int(2e4)  # evaluate the agent per training steps

    def init_before_training(self):
        if self.cwd is None:  # set cwd (current working directory) for saving model
            self.cwd = f'./{self.env_name}_{self.agent_class.__name__[5:]}'
        os.makedirs(self.cwd, exist_ok=True)


def get_gym_env_args(env, if_print: bool) -> dict:
    if {'unwrapped', 'observation_space', 'action_space', 'spec'}.issubset(dir(env)):  # isinstance(env, gym.Env):
        env_name = env.unwrapped.spec.id
        state_shape = env.observation_space.shape
        state_dim = state_shape[0] if len(state_shape) == 1 else state_shape  # sometimes state_dim is a list

        if_discrete = isinstance(env.action_space, gym.spaces.Discrete)
        if if_discrete:  # make sure it is discrete action space
            action_dim = env.action_space.n
        elif isinstance(env.action_space, gym.spaces.Box):  # make sure it is continuous action space
            action_dim = env.action_space.shape[0]

    env_args = {'env_name': env_name, 'state_dim': state_dim, 'action_dim': action_dim, 'if_discrete': if_discrete}
    print(f"env_args = {repr(env_args)}") if if_print else None
    return env_args


def kwargs_filter(function, kwargs: dict) -> dict:
    import inspect
    sign = inspect.signature(function).parameters.values()
    sign = {val.name for val in sign}
    common_args = sign.intersection(kwargs.keys())
    return {key: kwargs[key] for key in common_args}  # filtered kwargs


def build_env(env_class=None, env_args=None):
    if env_class.__module__ == 'gym.envs.registration':  # special rule
        env = env_class(id=env_args['env_name'])
    else:
        env = env_class(**kwargs_filter(env_class.__init__, env_args.copy()))
    for attr_str in ('env_name', 'state_dim', 'action_dim', 'if_discrete'):
        setattr(env, attr_str, env_args[attr_str])
    return env


class AgentBase:
    def __init__(self, net_dims: [int], state_dim: int, action_dim: int, gpu_id: int = 0, args: Config = Config()):
        self.state_dim = state_dim
        self.action_dim = action_dim

        self.gamma = args.gamma
        self.batch_size = args.batch_size
        self.repeat_times = args.repeat_times
        self.reward_scale = args.reward_scale
        self.soft_update_tau = args.soft_update_tau

        self.states = None  # assert self.states == (1, state_dim)
        self.device = device if device.type == 'cpu' else torch.device(f"cuda:{gpu_id}" if (torch.cuda.is_available() and (gpu_id >= 0)) else "cpu")

        act_class = getattr(self, "act_class", None)
        cri_class = getattr(self, "cri_class", None)
        self.act = self.act_target = act_class(net_dims, state_dim, action_dim).to(self.device)
        self.cri = self.cri_target = cri_class(net_dims, state_dim, action_dim).to(self.device) \
            if cri_class else self.act

        self.act_optimizer = torch.optim.Adam(self.act.parameters(), args.learning_rate)
        self.cri_optimizer = torch.optim.Adam(self.cri.parameters(), args.learning_rate) \
            if cri_class else self.act_optimizer

        self.criterion = torch.nn.SmoothL1Loss()

    @staticmethod
    def optimizer_update(optimizer, objective: Tensor):
        optimizer.zero_grad()
        objective.backward()
        optimizer.step()

    @staticmethod
    def soft_update(target_net: torch.nn.Module, current_net: torch.nn.Module, tau: float):
        for tar, cur in zip(target_net.parameters(), current_net.parameters()):
            tar.data.copy_(cur.data * tau + tar.data * (1.0 - tau))


class AgentPPO(AgentBase):
    def __init__(self, net_dims: [int], state_dim: int, action_dim: int, gpu_id: int = 0, args: Config = Config()):
        self.if_off_policy = False
        self.act_class = getattr(self, "act_class", ActorPPO)
        self.cri_class = getattr(self, "cri_class", CriticPPO)
        AgentBase.__init__(self, net_dims, state_dim, action_dim, gpu_id, args)

        self.ratio_clip = getattr(args, "ratio_clip", 0.25)  # `ratio.clamp(1 - clip, 1 + clip)`
        self.lambda_gae_adv = getattr(args, "lambda_gae_adv", 0.95)  # could be 0.80~0.99
        self.lambda_entropy = getattr(args, "lambda_entropy", 0.01)  # could be 0.00~0.10
        self.lambda_entropy = torch.tensor(self.lambda_entropy, dtype=torch.float32, device=self.device)

    def explore_env(self, env, horizon_len: int) -> [Tensor]:
        states = torch.zeros((horizon_len, self.state_dim), dtype=torch.float32).to(self.device)
        actions = torch.zeros((horizon_len, self.action_dim), dtype=torch.float32).to(self.device)
        logprobs = torch.zeros(horizon_len, dtype=torch.float32).to(self.device)
        rewards = torch.zeros(horizon_len, dtype=torch.float32).to(self.device)
        dones = torch.zeros(horizon_len, dtype=torch.bool).to(self.device)

        ary_state = self.states[0]

        get_action = self.act.get_action
        convert = self.act.convert_action_for_env
        for i in range(horizon_len):
            state = torch.as_tensor(ary_state, dtype=torch.float32, device=self.device)
            action, logprob = [t.squeeze(0) for t in get_action(state.unsqueeze(0))[:2]]

            ary_action = convert(action).detach().cpu().numpy()
            ary_state, reward, done, _, _ = env.step(ary_action)
            if done:
                ary_state, _ = env.reset()

            states[i] = state
            actions[i] = action
            logprobs[i] = logprob
            rewards[i] = reward
            dones[i] = done

        self.states[0] = ary_state
        rewards = (rewards * self.reward_scale).unsqueeze(1)
        undones = (1 - dones.type(torch.float32)).unsqueeze(1)
        return states, actions, logprobs, rewards, undones

    def update_net(self, buffer) -> [float]:
        with torch.no_grad():
            states, actions, logprobs, rewards, undones = buffer
            buffer_size = states.shape[0]

            '''get advantages reward_sums'''
            bs = 2 ** 10  # set a smaller 'batch_size' when out of GPU memory.
            values = [self.cri(states[i:i + bs]) for i in range(0, buffer_size, bs)]
            values = torch.cat(values, dim=0).squeeze(1)  # values.shape == (buffer_size, )

            advantages = self.get_advantages(rewards, undones, values)  # advantages.shape == (buffer_size, )
            reward_sums = advantages + values  # reward_sums.shape == (buffer_size, )
            del rewards, undones, values

            advantages = (advantages - advantages.mean()) / (advantages.std(dim=0) + 1e-5)
        assert logprobs.shape == advantages.shape == reward_sums.shape == (buffer_size,)

        '''update network'''
        obj_critics = 0.0
        obj_actors = 0.0

        update_times = int(buffer_size * self.repeat_times / self.batch_size)
        assert update_times >= 1
        for _ in range(update_times):
            indices = torch.randint(buffer_size, size=(self.batch_size,), requires_grad=False)
            state = states[indices]
            action = actions[indices]
            logprob = logprobs[indices]
            advantage = advantages[indices]
            reward_sum = reward_sums[indices]

            value = self.cri(state).squeeze(1)  # critic network predicts the reward_sum (Q value) of state
            obj_critic = self.criterion(value, reward_sum)
            self.optimizer_update(self.cri_optimizer, obj_critic)

            new_logprob, obj_entropy = self.act.get_logprob_entropy(state, action)
            ratio = (new_logprob - logprob.detach()).exp()
            surrogate1 = advantage * ratio
            surrogate2 = advantage * ratio.clamp(1 - self.ratio_clip, 1 + self.ratio_clip)
            obj_surrogate = torch.min(surrogate1, surrogate2).mean()

            obj_actor = obj_surrogate + obj_entropy.mean() * self.lambda_entropy
            self.optimizer_update(self.act_optimizer, -obj_actor)

            obj_critics += obj_critic.item()
            obj_actors += obj_actor.item()
        a_std_log = getattr(self.act, 'a_std_log', torch.zeros(1)).mean()
        return obj_critics / update_times, obj_actors / update_times, a_std_log.item()

    def get_advantages(self, rewards: Tensor, undones: Tensor, values: Tensor) -> Tensor:
        advantages = torch.empty_like(values)  # advantage value

        masks = undones * self.gamma
        horizon_len = rewards.shape[0]

        next_state = torch.tensor(self.states, dtype=torch.float32).to(self.device)
        next_value = self.cri(next_state).detach()[0, 0]

        advantage = 0  # last_gae_lambda
        for t in range(horizon_len - 1, -1, -1):
            delta = rewards[t] + masks[t] * next_value - values[t]
            advantages[t] = advantage = delta + masks[t] * self.lambda_gae_adv * advantage
            next_value = values[t]
        return advantages


class PendulumEnv(gym.Wrapper):  # a demo of custom gym env
    def __init__(self):
        gym.logger.set_level(40)  # Block warning
        gym_env_name = "Pendulum-v0" if gym.__version__ < '0.18.0' else "Pendulum-v1"
        super().__init__(env=gym.make(gym_env_name))

        '''the necessary env information when you design a custom env'''
        self.env_name = gym_env_name  # the name of this env.
        self.state_dim = self.observation_space.shape[0]  # feature number of state
        self.action_dim = self.action_space.shape[0]  # feature number of action
        self.if_discrete = False  # discrete action or continuous action

    def reset(self) -> np.ndarray:  # reset the agent in env
        resetted_env, _ = self.env.reset()
        return resetted_env

    def step(self, action: np.ndarray) -> (np.ndarray, float, bool, dict):  # agent interacts in env
        # We suggest that adjust action space to (-1, +1) when designing a custom env.
        state, reward, done, info_dict, _ = self.env.step(action * 2)
        return state.reshape(self.state_dim), float(reward), done, info_dict


def train_agent(args: Config):
    args.init_before_training()

    env = build_env(args.env_class, args.env_args)
    agent = args.agent_class(args.net_dims, args.state_dim, args.action_dim, gpu_id=args.gpu_id, args=args)

    new_env, _ = env.reset()
    agent.states = new_env[np.newaxis, :]

    evaluator = Evaluator(eval_env=build_env(args.env_class, args.env_args),
                          eval_per_step=args.eval_per_step,
                          eval_times=args.eval_times,
                          cwd=args.cwd)
    torch.set_grad_enabled(False)
    while True: # start training
        buffer_items = agent.explore_env(env, args.horizon_len)

        torch.set_grad_enabled(True)
        logging_tuple = agent.update_net(buffer_items)
        torch.set_grad_enabled(False)

        evaluator.evaluate_and_save(agent.act, args.horizon_len, logging_tuple)
        if (evaluator.total_step > args.break_step) or os.path.exists(f"{args.cwd}/stop"):
            torch.save(agent.act.state_dict(), args.cwd + '/actor.pth')
            break  # stop training when reach `break_step` or `mkdir cwd/stop`


def render_agent(env_class, env_args: dict, net_dims: [int], agent_class, actor_path: str, render_times: int = 8):
    env = build_env(env_class, env_args)

    state_dim = env_args['state_dim']
    action_dim = env_args['action_dim']
    gpu_id = 0 if device.type == 'cuda' else -1
    agent = agent_class(net_dims, state_dim, action_dim, gpu_id=gpu_id)
    actor = agent.act

    print(f"| render and load actor from: {actor_path}")
    load_device = device
    actor.load_state_dict(torch.load(actor_path, map_location=load_device))
    for i in range(render_times):
        cumulative_reward, episode_step = get_rewards_and_steps(env, actor, if_render=True)
        print(f"|{i:4}  cumulative_reward {cumulative_reward:9.3f}  episode_step {episode_step:5.0f}")


class Evaluator:
    def __init__(self, eval_env, eval_per_step: int = 1e4, eval_times: int = 8, cwd: str = '.'):
        self.cwd = cwd
        self.env_eval = eval_env
        self.eval_step = 0
        self.total_step = 0
        self.start_time = time.time()
        self.eval_times = eval_times  # number of times that get episodic cumulative return
        self.eval_per_step = eval_per_step  # evaluate the agent per training steps

        self.recorder = []
        print(f"\n| `step`: Number of samples, or total training steps, or running times of `env.step()`."
              f"\n| `time`: Time spent from the start of training to this moment."
              f"\n| `avgR`: Average value of cumulative rewards, which is the sum of rewards in an episode."
              f"\n| `stdR`: Standard dev of cumulative rewards, which is the sum of rewards in an episode."
              f"\n| `avgS`: Average of steps in an episode."
              f"\n| `objC`: Objective of Critic network. Or call it loss function of critic network."
              f"\n| `objA`: Objective of Actor network. It is the average Q value of the critic network."
              f"\n| {'step':>8}  {'time':>8}  | {'avgR':>8}  {'stdR':>6}  {'avgS':>6}  | {'objC':>8}  {'objA':>8}")

    def evaluate_and_save(self, actor, horizon_len: int, logging_tuple: tuple):
        self.total_step += horizon_len
        if self.eval_step + self.eval_per_step > self.total_step:
            return
        self.eval_step = self.total_step

        rewards_steps_ary = [get_rewards_and_steps(self.env_eval, actor) for _ in range(self.eval_times)]
        rewards_steps_ary = np.array(rewards_steps_ary, dtype=np.float32)
        avg_r = rewards_steps_ary[:, 0].mean()  # average of cumulative rewards
        std_r = rewards_steps_ary[:, 0].std()  # std of cumulative rewards
        avg_s = rewards_steps_ary[:, 1].mean()  # average of steps in an episode

        used_time = time.time() - self.start_time
        self.recorder.append((self.total_step, used_time, avg_r))

        print(f"| {self.total_step:8.2e}  {used_time:8.0f}  "
              f"| {avg_r:8.2f}  {std_r:6.2f}  {avg_s:6.0f}  "
              f"| {logging_tuple[0]:8.2f}  {logging_tuple[1]:8.2f}")


def get_rewards_and_steps(env, actor, if_render: bool = False) -> (float, int):  # cumulative_rewards and episode_steps
    device = next(actor.parameters()).device  # net.parameters() is a Python generator.

    state, _ = env.reset()
    episode_steps = 0
    cumulative_returns = 0.0  # sum of rewards in an episode
    for episode_steps in range(12345):
        tensor_state = torch.as_tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        tensor_action = actor(tensor_state)
        action = tensor_action.detach().cpu().numpy()[0]  # not need detach(), because using torch.no_grad() outside
        state, reward, done, _, _ = env.step(action)
        cumulative_returns += reward

        if if_render:
            env.render()
        if done:
            break
    return cumulative_returns, episode_steps + 1

CUDA kernel incompatible with Quadro P620.
Falling back to CPU.


## DRL Agent Class

In [28]:
from __future__ import annotations

import torch
# from elegantrl.agents import AgentA2C

MODELS = {"ppo": AgentPPO}
OFF_POLICY_MODELS = ["ddpg", "td3", "sac"]
ON_POLICY_MODELS = ["ppo"]
# MODEL_KWARGS = {x: config.__dict__[f"{x.upper()}_PARAMS"] for x in MODELS.keys()}
#
# NOISE = {
#     "normal": NormalActionNoise,
#     "ornstein_uhlenbeck": OrnsteinUhlenbeckActionNoise,
# }


class DRLAgent:
    """Implementations of DRL algorithms
    Attributes
    ----------
        env: gym environment class
            user-defined class
    Methods
    -------
        get_model()
            setup DRL algorithms
        train_model()
            train DRL algorithms in a train dataset
            and output the trained model
        DRL_prediction()
            make a prediction in a test dataset and get results
    """

    def __init__(self, env, price_array, tech_array, turbulence_array, account_config=None):
        self.env = env
        self.price_array = price_array
        self.tech_array = tech_array
        self.turbulence_array = turbulence_array
        self.account_config = account_config or {}

    def get_model(self, model_name, model_kwargs):
        env_config = {
            "price_array": self.price_array,
            "tech_array": self.tech_array,
            "turbulence_array": self.turbulence_array,
            "if_train": True,
        }
        environment = self.env(config=env_config, **self.account_config)
        env_args = {'config': env_config,
              'env_name': environment.env_name,
              'state_dim': environment.state_dim,
              'action_dim': environment.action_dim,
              'if_discrete': False,
              **self.account_config}
        agent = MODELS[model_name]
        if model_name not in MODELS:
            raise NotImplementedError("NotImplementedError")
        model = Config(agent_class=agent, env_class=self.env, env_args=env_args)
        model.if_off_policy = model_name in OFF_POLICY_MODELS
        if model_kwargs is not None:
            try:
                model.learning_rate = model_kwargs["learning_rate"]
                model.batch_size = model_kwargs["batch_size"]
                model.gamma = model_kwargs["gamma"]
                model.seed = model_kwargs["seed"]
                model.net_dims = model_kwargs["net_dimension"]
                model.target_step = model_kwargs["target_step"]
                model.eval_gap = model_kwargs["eval_gap"]
                model.eval_times = model_kwargs["eval_times"]
            except BaseException:
                raise ValueError(
                    "Fail to read arguments, please check 'model_kwargs' input."
                )
        return model

    def train_model(self, model, cwd, total_timesteps=5000):
        model.cwd = cwd
        model.break_step = total_timesteps
        train_agent(model)

    @staticmethod
    def DRL_prediction(model_name, cwd, net_dimension, environment):
        if model_name not in MODELS:
            raise NotImplementedError("NotImplementedError")
        agent_class = MODELS[model_name]
        environment.env_num = 1
        gpu_id = 0 if (device.type == 'cuda' and torch.cuda.is_available()) else -1
        agent = agent_class(net_dimension, environment.state_dim, environment.action_dim, gpu_id=gpu_id)
        actor = agent.act
        # load agent
        try:
            cwd = cwd + '/actor.pth'
            print(f"| load actor from: {cwd}")
            load_device = device
            actor.load_state_dict(torch.load(cwd, map_location=load_device))
            act = actor
            act_device = agent.device
        except BaseException:
            raise ValueError("Fail to load agent!")

        # test on the testing env
        _torch = torch
        state, _ = environment.reset()
        episode_returns = []  # the cumulative_return / initial_account
        episode_total_assets = [environment.initial_total_asset]
        with _torch.no_grad():
            for i in range(environment.max_step):
                s_tensor = _torch.as_tensor((state,), device=act_device)
                a_tensor = act(s_tensor)  # action_tanh = act.forward()
                action = (
                    a_tensor.detach().cpu().numpy()[0]
                )  # not need detach(), because with torch.no_grad() outside
                state, reward, done, _, _ = environment.step(action)

                total_asset = (
                    environment.amount
                    + (
                        environment.price_ary[environment.day] * environment.stocks
                    ).sum()
                )
                episode_total_assets.append(total_asset)
                episode_return = total_asset / environment.initial_total_asset
                episode_returns.append(episode_return)
                if done:
                    break
        print("Test Finished!")
        # return episode total_assets on testing data
        print("episode_return", episode_return)
        return episode_total_assets

## Train & Test Functions

In [47]:
from __future__ import annotations

from finrl.config import ERL_PARAMS
from finrl.config import INDICATORS
from finrl.config import RLlib_PARAMS
from finrl.config import SAC_PARAMS
from finrl.config import TRAIN_END_DATE
from finrl.config import TRAIN_START_DATE
from finrl.config_tickers import DOW_30_TICKER
from finrl.meta.data_processor import DataProcessor

# === TRANSACTION COST CONFIGURATION ===
TRANSACTION_COSTS = {
    "buy_cost_pct": 0.001,   # 0.1% for spread + slippage
    "sell_cost_pct": 0.001,
}

# === DATA CACHE ===
# Download once per unique date range, reuse across all HPO/OOS iterations.
_data_cache = {}

def download_and_cache(start_date, end_date, ticker_list, data_source,
                       time_interval, technical_indicator_list, if_vix=True,
                       **kwargs):
    """Download & process data once; return cached arrays on repeat calls."""
    cache_key = (start_date, end_date)
    if cache_key in _data_cache:
        print(f"  [cache hit] {start_date} → {end_date}")
        return _data_cache[cache_key]

    print(f"  [downloading] {start_date} → {end_date} ...")
    dp = DataProcessor(data_source, **kwargs)
    data = dp.download_data(ticker_list, start_date, end_date, time_interval)
    data = dp.clean_data(data)
    data = dp.add_technical_indicator(data, technical_indicator_list)
    if if_vix:
        data = dp.add_vix(data)
    else:
        data = dp.add_turbulence(data)
    price_array, tech_array, turbulence_array = dp.df_to_array(data, if_vix)
    _data_cache[cache_key] = (price_array, tech_array, turbulence_array)
    return price_array, tech_array, turbulence_array


def train(
    start_date,
    end_date,
    ticker_list,
    data_source,
    time_interval,
    technical_indicator_list,
    drl_lib,
    env,
    model_name,
    if_vix=True,
    **kwargs,
):
    # Use cached data if available, otherwise download
    cached = kwargs.pop("cached_arrays", None)
    if cached is not None:
        price_array, tech_array, turbulence_array = cached
    else:
        price_array, tech_array, turbulence_array = download_and_cache(
            start_date, end_date, ticker_list, data_source,
            time_interval, technical_indicator_list, if_vix, **kwargs
        )
    
    # Include transaction costs
    env_config = {
        "price_array": price_array,
        "tech_array": tech_array,
        "turbulence_array": turbulence_array,
        "if_train": True,
        **TRANSACTION_COSTS,
    }

    # Account configuration (initial_capital, max_stock)
    account_config = kwargs.get("account_config", {})
    env_instance = env(config=env_config, **account_config)

    # read parameters
    cwd = kwargs.get("cwd", "./" + str(model_name))

    if drl_lib == "elegantrl":
        DRLAgent_erl = DRLAgent
        break_step = kwargs.get("break_step", 1e6)
        erl_params = kwargs.get("erl_params")
        agent = DRLAgent_erl(
            env=env,
            price_array=price_array,
            tech_array=tech_array,
            turbulence_array=turbulence_array,
            account_config=account_config,
        )
        model = agent.get_model(model_name, model_kwargs=erl_params)
        trained_model = agent.train_model(
            model=model, cwd=cwd, total_timesteps=break_step
        )

In [48]:
from __future__ import annotations

from finrl.config import INDICATORS
from finrl.config import RLlib_PARAMS
from finrl.config import TEST_END_DATE
from finrl.config import TEST_START_DATE
from finrl.config_tickers import DOW_30_TICKER

def test(
    start_date,
    end_date,
    ticker_list,
    data_source,
    time_interval,
    technical_indicator_list,
    drl_lib,
    env,
    model_name,
    if_vix=True,
    **kwargs,
):
    # Use cached data if available, otherwise download
    cached = kwargs.pop("cached_arrays", None)
    if cached is not None:
        price_array, tech_array, turbulence_array = cached
    else:
        from finrl.meta.data_processor import DataProcessor
        price_array, tech_array, turbulence_array = download_and_cache(
            start_date, end_date, ticker_list, data_source,
            time_interval, technical_indicator_list, if_vix, **kwargs
        )

    # Include transaction costs
    # Account configuration (initial_capital, max_stock)
    account_config = kwargs.get("account_config", {})
    env_config = {
        "price_array": price_array,
        "tech_array": tech_array,
        "turbulence_array": turbulence_array,
        "if_train": False,
        **TRANSACTION_COSTS,
    }
    env_instance = env(config=env_config, **account_config)

    net_dimension = kwargs.get("net_dimension", 2**7)
    cwd = kwargs.get("cwd", "./" + str(model_name))
    print("price_array: ", len(price_array))

    if drl_lib == "elegantrl":
        DRLAgent_erl = DRLAgent
        episode_total_assets = DRLAgent_erl.DRL_prediction(
            model_name=model_name,
            cwd=cwd,
            net_dimension=net_dimension,
            environment=env_instance,
        )
        return episode_total_assets


# === CORRECTED METRIC CALCULATIONS ===
def calculate_metrics(account_values, periods_per_day=390):
    """
    Calculate properly annualized Sharpe ratio and correct max drawdown.
    """
    account_values = np.array(account_values)
    
    total_return = (account_values[-1] / account_values[0]) - 1
    
    # Resample to daily for meaningful Sharpe
    daily_values = account_values[::periods_per_day]
    if len(daily_values) < 2:
        daily_values = account_values
    
    daily_returns = np.diff(daily_values) / daily_values[:-1]
    
    # Annualized Sharpe
    if len(daily_returns) > 1 and np.std(daily_returns) > 1e-10:
        sharpe_ratio = np.mean(daily_returns) / np.std(daily_returns) * np.sqrt(252)
    else:
        sharpe_ratio = 0.0
    
    # Correct max drawdown
    running_max = np.maximum.accumulate(account_values)
    drawdowns = (account_values - running_max) / running_max
    max_drawdown = np.min(drawdowns)
    
    # Calmar ratio
    trading_days = len(account_values) / periods_per_day
    annualized_return = (1 + total_return) ** (252 / max(trading_days, 1)) - 1
    calmar_ratio = annualized_return / abs(max_drawdown) if max_drawdown != 0 else 0
    
    return {
        'total_return': total_return,
        'sharpe_ratio': sharpe_ratio,
        'max_drawdown': max_drawdown,
        'calmar_ratio': calmar_ratio,
        'trading_days': trading_days
    }

## Import Dow Jones 30 Symbols

In [49]:
# Remove WBA since it has no data available from Alpaca
ticker_list = [ticker for ticker in DOW_30_TICKER if ticker != 'WBA']
action_dim = len(ticker_list)
print(f"Using {action_dim} stocks (removed WBA - no data available)")
print(f"Ticker list: {ticker_list}")

Using 29 stocks (removed WBA - no data available)
Ticker list: ['AXP', 'AMGN', 'AAPL', 'BA', 'CAT', 'CSCO', 'CVX', 'GS', 'HD', 'HON', 'IBM', 'INTC', 'JNJ', 'KO', 'JPM', 'MCD', 'MMM', 'MRK', 'MSFT', 'NKE', 'PG', 'TRV', 'UNH', 'CRM', 'VZ', 'V', 'WMT', 'DIS', 'DOW']


In [8]:
print(ticker_list)

['AXP', 'AMGN', 'AAPL', 'BA', 'CAT', 'CSCO', 'CVX', 'GS', 'HD', 'HON', 'IBM', 'INTC', 'JNJ', 'KO', 'JPM', 'MCD', 'MMM', 'MRK', 'MSFT', 'NKE', 'PG', 'TRV', 'UNH', 'CRM', 'VZ', 'V', 'WMT', 'DIS', 'DOW']


In [9]:
print(INDICATORS)

['macd', 'boll_ub', 'boll_lb', 'rsi_30', 'cci_30', 'dx_30', 'close_30_sma', 'close_60_sma']


## Calculate the DRL state dimension manually for paper trading

In [50]:
# amount + (turbulence, turbulence_bool) + (price, shares, cd (holding time)) * stock_dim + tech_dim
# Updated to 29 stocks (removed WBA)
state_dim = 1 + 2 + 3 * action_dim + len(INDICATORS) * action_dim
print(f"State dimension: {state_dim} (for {action_dim} stocks)")

State dimension: 322 (for 29 stocks)


In [ ]:
state_dim

In [ ]:
action_dim

## Get the API Keys Ready

In [51]:
import os
from dotenv import load_dotenv
load_dotenv()

API_KEY = os.environ.get('ALPACA_API_KEY')
API_SECRET = os.environ.get('ALPACA_API_SECRET')
API_BASE_URL = os.environ.get('ALPACA_API_BASE_URL', 'https://paper-api.alpaca.markets')
data_url = os.environ.get('ALPACA_DATA_URL', 'wss://data.alpaca.markets')

env = StockTradingEnv

## Account Size Configuration

Configure `INITIAL_CAPITAL` and `MAX_STOCK` for your trading account tier.

`episode_return = total_asset / initial_capital` is ratio-based and account-size invariant.

**Note:** `reward_scaling` (env-level, `StockTradingEnv` from FinRL, default `2**-11`) scales per-step rewards. `reward_scale` (agent-level, local `Config` class, default `1.0`) further scales rewards in the PPO update. PPO normalizes advantages, so neither materially affects training.

The key parameters for different account sizes are:
- **`INITIAL_CAPITAL`**: Starting cash for training environment (`StockTradingEnv`)
- **`MAX_STOCK`**: Max shares per trade action (scales with account size)

In [52]:
# === ACCOUNT SIZE CONFIGURATION ===
# Uncomment ONE preset below, or set custom values.
#
# StockTradingEnv uses hardcoded 2**-12 to normalize cash in get_state().
# That targets $1M -> ~244. CASH_NORM auto-scales so the state feature
# stays ~244 regardless of account size.
# The key parameters are INITIAL_CAPITAL and MAX_STOCK.

# --- Preset: $1M (original default) ---
INITIAL_CAPITAL = 1_000_000
MAX_STOCK = 1e2

# --- Preset: $100K ---
#INITIAL_CAPITAL = 100_000
#MAX_STOCK = 1e2

# --- Preset: $50K ---
#INITIAL_CAPITAL = 50_000
#MAX_STOCK = 1e2

# --- Preset: $10K ---
#INITIAL_CAPITAL = 10_000
#MAX_STOCK = 10

# Auto-scale cash normalization: 2**-12 was designed for $1M (~244).
# Scale so that INITIAL_CAPITAL * CASH_NORM ~ 244 for any account size.
CASH_NORM = 2**-12 * (1_000_000 / INITIAL_CAPITAL)

# Tier suffix for HPO directories — auto-derived from INITIAL_CAPITAL
# e.g. 1_000_000 → "_1m", 100_000 → "_100k", 50_000 → "_50k", 10_000 → "_10k"
if INITIAL_CAPITAL >= 1_000_000:
    TIER_SUFFIX = f"_{INITIAL_CAPITAL // 1_000_000}m"
else:
    TIER_SUFFIX = f"_{INITIAL_CAPITAL // 1_000}k"

ACCOUNT_CONFIG = {
    "initial_capital": INITIAL_CAPITAL,
    "max_stock": MAX_STOCK,
}

print("=" * 60)
print("ACCOUNT CONFIGURATION")
print("=" * 60)
print(f"Initial Capital:  ${INITIAL_CAPITAL:>12,.0f}")
print(f"Max Stock/Trade:  {MAX_STOCK:>12.0f}")
print(f"Cash Norm:        {CASH_NORM:.10f}  (cash * norm ~ {INITIAL_CAPITAL * CASH_NORM:.1f})")
print(f"Tier Suffix:      {TIER_SUFFIX}")
print("=" * 60)

ACCOUNT CONFIGURATION
Initial Capital:  $   1,000,000
Max Stock/Trade:           100
Cash Norm:        0.0002441406  (cash * norm ~ 244.1)
Tier Suffix:      _1m


# Improved Hyperparameter Optimization with Walk-Forward Validation

Key improvements over original approach:
1. **Walk-Forward Validation**: Test on multiple time windows to ensure robustness
2. **Corrected Sharpe Ratio**: Properly annualized using daily returns
3. **Corrected Max Drawdown**: Fixed calculation error
4. **Transaction Costs**: Included in training environment
5. **Smaller Network Options**: To reduce overfitting risk
6. **Longer Training Period**: 4+ months per window

## Walk-Forward Validation Windows

Instead of a single train/test split, we use multiple rolling windows to:
- Reduce selection bias
- Test across different market regimes
- Get statistically meaningful results

In [36]:
import itertools
import json
from datetime import datetime as dt

# === WALK-FORWARD VALIDATION WINDOWS ===
# Each window: (train_start, train_end, val_start, val_end)
# Using ~4 months training, ~1 month validation per window
VALIDATION_WINDOWS = [
    ('2025-02-01', '2025-05-31', '2025-06-01', '2025-06-30'),  # Window 1
    ('2025-04-01', '2025-07-31', '2025-08-01', '2025-08-31'),  # Window 2  
    ('2025-06-01', '2025-09-30', '2025-10-01', '2025-10-31'),  # Window 3
]

print(f"Using {len(VALIDATION_WINDOWS)} walk-forward validation windows:")
for i, (ts, te, vs, ve) in enumerate(VALIDATION_WINDOWS):
    print(f"  Window {i+1}: Train {ts} to {te} → Validate {vs} to {ve}")

# === IMPROVED PARAMETER GRID ===
# Smaller networks to reduce overfitting
# Lower gamma values for shorter trading horizons
param_grid = {
    'learning_rate': [1e-5, 3e-5, 1e-4],
    'gamma': [0.90, 0.95, 0.97],  # Lower gamma for intraday trading
    'net_dimension': [[32, 16], [64, 32], [128, 64]],  # Smaller networks
    'batch_size': [256, 512, 1024]  # Smaller batches for more updates
}

# Generate all combinations
param_combinations = list(itertools.product(
    param_grid['learning_rate'],
    param_grid['gamma'],
    param_grid['net_dimension'],
    param_grid['batch_size']
))

print(f"\nTotal hyperparameter combinations: {len(param_combinations)}")
print(f"Total experiments (combinations × windows): {len(param_combinations) * len(VALIDATION_WINDOWS)}")

Using 3 walk-forward validation windows:
  Window 1: Train 2025-02-01 to 2025-05-31 → Validate 2025-06-01 to 2025-06-30
  Window 2: Train 2025-04-01 to 2025-07-31 → Validate 2025-08-01 to 2025-08-31
  Window 3: Train 2025-06-01 to 2025-09-30 → Validate 2025-10-01 to 2025-10-31

Total hyperparameter combinations: 81
Total experiments (combinations × windows): 243


## Run Grid Search (WARNING: Time-consuming!)

In [37]:
# === CHECKPOINT REPAIR UTILITY ===
# Run this cell ONLY if you want to inspect/clean the checkpoint manually.
# The main grid search cell now auto-strips failed experiments on resume,
# so this is typically not needed. Use it to diagnose checkpoint issues.

import os
import pickle
import pandas as pd

CHECKPOINT_DIR = f'./hpo_checkpoints{TIER_SUFFIX}'
CHECKPOINT_FILE = os.path.join(CHECKPOINT_DIR, 'hpo_checkpoint.pkl')

def repair_checkpoint():
    """Remove failed experiments from checkpoint, keeping only successful ones.
    Also validates experiment IDs against the current param grid."""
    if not os.path.exists(CHECKPOINT_FILE):
        print("❌ No checkpoint file found!")
        return
    
    with open(CHECKPOINT_FILE, 'rb') as f:
        checkpoint = pickle.load(f)
    
    results = checkpoint['results']
    completed = checkpoint['completed_experiments']
    
    # Count successes and failures
    successful = [r for r in results if 'error' not in r]
    failed = [r for r in results if 'error' in r]
    
    print(f"Current checkpoint status:")
    print(f"  Total experiments marked complete: {len(completed)}")
    print(f"  Successful results: {len(successful)}")
    print(f"  Failed results: {len(failed)}")
    
    # Cross-check against current grid (requires param_combinations from cell above)
    try:
        valid_exp_ids = set()
        for lr_v, gamma_v, net_dim_v, bs_v in param_combinations:
            ek = f"lr{lr_v:.0e}_g{gamma_v}_net{'x'.join(map(str, net_dim_v))}_bs{bs_v}"
            for wi in range(len(VALIDATION_WINDOWS)):
                valid_exp_ids.add(f"{ek}_window{wi+1}")
        
        stale = set(completed) - valid_exp_ids
        if stale:
            print(f"  ⚠️  Stale IDs not in current grid: {len(stale)}")
            for s in sorted(stale)[:5]:
                print(f"      {s}")
            if len(stale) > 5:
                print(f"      ... and {len(stale)-5} more")
        else:
            print(f"  ✅ All completed IDs match current grid")
    except NameError:
        print("  ℹ️  Run param grid cell first to enable grid validation")
    
    if len(failed) == 0:
        print("\n✅ No failed experiments to remove!")
        return
    
    # Get exp_ids for successful experiments only
    successful_ids = set()
    for r in successful:
        exp_key = r['exp_key']
        window = r['window']
        exp_id = f"{exp_key}_window{window}"
        successful_ids.add(exp_id)
    
    # Rebuild aggregated_results from successful only
    from collections import defaultdict
    new_aggregated = defaultdict(lambda: {
        'sharpe_ratios': [],
        'returns': [],
        'max_drawdowns': [],
        'successful_windows': 0
    })
    
    for r in successful:
        exp_key = r['exp_key']
        new_aggregated[exp_key]['sharpe_ratios'].append(r['sharpe_ratio'])
        new_aggregated[exp_key]['returns'].append(r['total_return'])
        new_aggregated[exp_key]['max_drawdowns'].append(r['max_drawdown'])
        new_aggregated[exp_key]['successful_windows'] += 1
        new_aggregated[exp_key]['params'] = {
            'learning_rate': r['learning_rate'],
            'gamma': r['gamma'],
            'net_dimension': r['net_dimension'],
            'batch_size': r['batch_size']
        }
    
    # Create repaired checkpoint
    repaired = {
        'results': successful,
        'aggregated_results': dict(new_aggregated),
        'completed_experiments': list(successful_ids),
        'timestamp': checkpoint['timestamp'] + ' (REPAIRED)'
    }
    
    # Backup old checkpoint
    import shutil
    backup_file = CHECKPOINT_FILE + '.backup'
    shutil.copy2(CHECKPOINT_FILE, backup_file)
    print(f"\n📦 Old checkpoint backed up to: {backup_file}")
    
    # Save repaired checkpoint
    with open(CHECKPOINT_FILE, 'wb') as f:
        pickle.dump(repaired, f)
    
    print(f"✅ Checkpoint repaired!")
    print(f"   Kept {len(successful)} successful experiments")
    print(f"   Removed {len(failed)} failed experiments")
    print(f"\n🔄 Now re-run the grid search cell to resume from experiment {len(successful)+1}")

# Run the repair
repair_checkpoint()

Current checkpoint status:
  Total experiments marked complete: 243
  Successful results: 243
  Failed results: 0
  ✅ All completed IDs match current grid

✅ No failed experiments to remove!


In [ ]:
# Walk-Forward Grid Search with CHECKPOINTING + CONNECTION MONITORING
# This tests each hyperparameter combination across ALL validation windows
# and aggregates results to find the most robust configuration
# 
# CHECKPOINT FEATURE: Saves progress after each experiment. If the kernel crashes
# or disconnects, simply re-run this cell to resume from where you left off.
#
# CONNECTION MONITORING: Detects connection failures and exits gracefully instead
# of continuing to fail. Re-run the cell after connection is restored.

import os
import time
import json
import pickle
import socket
import requests
from collections import defaultdict

# === CHECKPOINT CONFIGURATION ===
CHECKPOINT_DIR = f'./hpo_checkpoints{TIER_SUFFIX}'
CHECKPOINT_FILE = os.path.join(CHECKPOINT_DIR, 'hpo_checkpoint.pkl')
RESULTS_BACKUP_FILE = os.path.join(CHECKPOINT_DIR, 'results_backup.csv')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# === CONNECTION MONITORING ===
CONSECUTIVE_FAILURE_LIMIT = 2  # Exit after this many consecutive failures
CONNECTION_CHECK_TIMEOUT = 10  # seconds

def check_internet_connection():
    """Check if we have internet connectivity."""
    try:
        # Try to connect to Alpaca API
        response = requests.get('https://api.alpaca.markets/v2/clock', 
                               headers={'APCA-API-KEY-ID': API_KEY, 
                                       'APCA-API-SECRET-KEY': API_SECRET},
                               timeout=CONNECTION_CHECK_TIMEOUT)
        return response.status_code == 200
    except:
        pass
    
    # Fallback: try common endpoints
    for host in ['8.8.8.8', '1.1.1.1']:
        try:
            socket.create_connection((host, 53), timeout=3)
            return True
        except:
            continue
    return False

def is_connection_error(error_msg):
    """Check if an error is likely due to connection issues."""
    connection_keywords = [
        'connection', 'timeout', 'timed out', 'network', 'unreachable',
        'refused', 'reset', 'socket', 'ssl', 'certificate', 'dns',
        'name resolution', 'temporarily unavailable', 'no route',
        'connection reset', 'broken pipe', 'eof', 'connection aborted'
    ]
    error_lower = str(error_msg).lower()
    return any(keyword in error_lower for keyword in connection_keywords)

def save_checkpoint(results, aggregated_results, completed_experiments):
    """Save current progress to checkpoint file."""
    checkpoint = {
        'results': results,
        'aggregated_results': dict(aggregated_results),
        'completed_experiments': completed_experiments,
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')
    }
    with open(CHECKPOINT_FILE, 'wb') as f:
        pickle.dump(checkpoint, f)
    
    # Also save results as CSV backup
    if results:
        results_df = pd.DataFrame([r for r in results if 'error' not in r])
        if len(results_df) > 0:
            results_df.to_csv(RESULTS_BACKUP_FILE, index=False)
    
    print(f"   💾 Checkpoint saved ({len(completed_experiments)} experiments complete)")

def load_checkpoint():
    """Load checkpoint if it exists."""
    if os.path.exists(CHECKPOINT_FILE):
        try:
            with open(CHECKPOINT_FILE, 'rb') as f:
                checkpoint = pickle.load(f)
            print(f"📂 Checkpoint found from {checkpoint['timestamp']}")
            print(f"   Resuming from {len(checkpoint['completed_experiments'])} completed experiments")
            return checkpoint
        except Exception as e:
            print(f"⚠️ Failed to load checkpoint: {e}")
            return None
    return None

# Pre-flight connection check
print("🔌 Checking connection to Alpaca API...")
if not check_internet_connection():
    print("❌ Cannot connect to Alpaca API!")
    print("   Please check your internet connection and try again.")
    raise ConnectionError("No connection to Alpaca API. Please restore connection and re-run.")
print("✅ Connection OK\n")

# Try to load existing checkpoint
checkpoint = load_checkpoint()

if checkpoint:
    # Only keep successful results — failed experiments will be retried
    results = [r for r in checkpoint['results'] if 'error' not in r]
    failed_in_checkpoint = [r for r in checkpoint['results'] if 'error' in r]
    
    # Reconstruct defaultdict from regular dict
    aggregated_results = defaultdict(lambda: {
        'sharpe_ratios': [],
        'returns': [],
        'max_drawdowns': [],
        'successful_windows': 0
    })
    for key, value in checkpoint['aggregated_results'].items():
        aggregated_results[key] = value
    
    # Build the set of valid experiment IDs from the current grid
    valid_exp_ids = set()
    for lr_v, gamma_v, net_dim_v, bs_v in param_combinations:
        ek = f"lr{lr_v:.0e}_g{gamma_v}_net{'x'.join(map(str, net_dim_v))}_bs{bs_v}"
        for wi in range(len(VALIDATION_WINDOWS)):
            valid_exp_ids.add(f"{ek}_window{wi+1}")
    
    # Only keep completed IDs that match the current grid (drop stale/orphan IDs)
    raw_completed = set(checkpoint['completed_experiments'])
    stale_ids = raw_completed - valid_exp_ids
    if stale_ids:
        print(f"⚠️  Dropping {len(stale_ids)} stale experiment IDs not in current grid")
    
    # Also drop IDs for experiments that had errors (so they get retried)
    failed_ids = set()
    for r in failed_in_checkpoint:
        fk = r['exp_key']
        fw = r['window']
        failed_ids.add(f"{fk}_window{fw}")
    if failed_ids:
        print(f"🔄 {len(failed_ids)} previously failed experiments will be retried")
    
    completed_experiments = (raw_completed & valid_exp_ids) - failed_ids
    print(f"✅ Resuming HPO - {len(completed_experiments)} successful experiments loaded\n")
else:
    results = []
    aggregated_results = defaultdict(lambda: {
        'sharpe_ratios': [],
        'returns': [],
        'max_drawdowns': [],
        'successful_windows': 0
    })
    completed_experiments = set()
    print("🆕 Starting fresh HPO run\n")

# Option 1: Test a subset (recommended for quick testing)
# test_combinations = param_combinations[:3]

# Option 2: Test all combinations
test_combinations = param_combinations

# Option 3: Random search - test N random combinations
# import random
# random.seed(312)
# N_RANDOM_TESTS = 27
# test_combinations = random.sample(param_combinations, min(N_RANDOM_TESTS, len(param_combinations)))

total_experiments = len(test_combinations) * len(VALIDATION_WINDOWS)
remaining = total_experiments - len(completed_experiments)

print(f"Testing {len(test_combinations)} combinations across {len(VALIDATION_WINDOWS)} windows")
print(f"Total experiments: {total_experiments}")
print(f"Already completed: {len(completed_experiments)}")
print(f"Remaining: {remaining}")

# === PRE-CACHE ALL DATA ===
# Download each unique date range once instead of 81× per window
print(f"\n📥 Pre-caching data for {len(VALIDATION_WINDOWS)} windows...")
_data_cache.clear()  # fresh start
for wi, (ts, te, vs, ve) in enumerate(VALIDATION_WINDOWS, 1):
    print(f"  Window {wi}: train {ts}→{te}, val {vs}→{ve}")
    download_and_cache(ts, te, ticker_list, 'alpaca', '1Min', INDICATORS,
                       True, API_KEY=API_KEY, API_SECRET=API_SECRET,
                       API_BASE_URL=API_BASE_URL)
    download_and_cache(vs, ve, ticker_list, 'alpaca', '1Min', INDICATORS,
                       True, API_KEY=API_KEY, API_SECRET=API_SECRET,
                       API_BASE_URL=API_BASE_URL)
print(f"✅ {len(_data_cache)} date ranges cached\n")

consecutive_failures = 0  # Track consecutive connection failures
graceful_exit = False

for lr, gamma, net_dim, bs in test_combinations:
    if graceful_exit:
        break
        
    exp_key = f"lr{lr:.0e}_g{gamma}_net{'x'.join(map(str, net_dim))}_bs{bs}"
    
    for window_idx, (train_start, train_end, val_start, val_end) in enumerate(VALIDATION_WINDOWS):
        if graceful_exit:
            break
        
        # Create unique experiment ID
        exp_id = f"{exp_key}_window{window_idx+1}"
        
        # Skip if already completed
        if exp_id in completed_experiments:
            continue
        
        exps_done = len(completed_experiments)
        exps_remaining = total_experiments - exps_done
        print(f"\n{'='*70}")
        print(f"Experiment {exps_done + 1}/{total_experiments} ({exps_remaining} remaining)")
        print(f"Config: LR={lr:.0e}, Gamma={gamma}, Net={net_dim}, Batch={bs}")
        print(f"Window {window_idx+1}: Train {train_start}→{train_end} | Val {val_start}→{val_end}")
        print(f"{'='*70}")
        
        exp_params = {
            "learning_rate": lr,
            "batch_size": bs,
            "gamma": gamma,
            "seed": 312,
            "net_dimension": net_dim,
            "target_step": 5000,
            "eval_gap": 30,
            "eval_times": 1
        }
        
        exp_cwd = f'./hpo_experiments{TIER_SUFFIX}/{exp_key}/window_{window_idx+1}'
        os.makedirs(exp_cwd, exist_ok=True)
        
        try:
            # Fetch cached arrays for this window
            train_arrays = _data_cache[(train_start, train_end)]
            val_arrays = _data_cache[(val_start, val_end)]

            # Train on training window
            train(
                start_date=train_start,
                end_date=train_end,
                ticker_list=ticker_list,
                data_source='alpaca',
                time_interval='1Min',
                technical_indicator_list=INDICATORS,
                drl_lib='elegantrl',
                env=env,
                model_name='ppo',
                if_vix=True,
                API_KEY=API_KEY,
                API_SECRET=API_SECRET,
                API_BASE_URL=API_BASE_URL,
                erl_params=exp_params,
                cwd=exp_cwd,
                break_step=1e5,
                account_config=ACCOUNT_CONFIG,
                cached_arrays=train_arrays,
            )
            
            # Test on validation window
            account_values = test(
                start_date=val_start,
                end_date=val_end,
                ticker_list=ticker_list,
                data_source='alpaca',
                time_interval='1Min',
                technical_indicator_list=INDICATORS,
                drl_lib='elegantrl',
                env=env,
                model_name='ppo',
                if_vix=True,
                API_KEY=API_KEY,
                API_SECRET=API_SECRET,
                API_BASE_URL=API_BASE_URL,
                cwd=exp_cwd,
                net_dimension=net_dim,
                account_config=ACCOUNT_CONFIG,
                cached_arrays=val_arrays,
            )
            
            # Use corrected metrics calculation
            metrics = calculate_metrics(account_values)
            
            result = {
                'learning_rate': lr,
                'gamma': gamma,
                'net_dimension': net_dim,
                'batch_size': bs,
                'window': window_idx + 1,
                'train_start': train_start,
                'train_end': train_end,
                'val_start': val_start,
                'val_end': val_end,
                'total_return': metrics['total_return'],
                'sharpe_ratio': metrics['sharpe_ratio'],
                'max_drawdown': metrics['max_drawdown'],
                'final_value': account_values[-1],
                'exp_key': exp_key
            }
            results.append(result)
            
            # Aggregate for this config
            aggregated_results[exp_key]['sharpe_ratios'].append(metrics['sharpe_ratio'])
            aggregated_results[exp_key]['returns'].append(metrics['total_return'])
            aggregated_results[exp_key]['max_drawdowns'].append(metrics['max_drawdown'])
            aggregated_results[exp_key]['successful_windows'] += 1
            aggregated_results[exp_key]['params'] = {
                'learning_rate': lr, 'gamma': gamma, 
                'net_dimension': net_dim, 'batch_size': bs
            }
            
            print(f"\n✅ Window {window_idx+1} Results:")
            print(f"   Total Return: {metrics['total_return']:.2%}")
            print(f"   Sharpe Ratio: {metrics['sharpe_ratio']:.3f}")
            print(f"   Max Drawdown: {metrics['max_drawdown']:.2%}")
            
            # Mark as completed and save checkpoint
            completed_experiments.add(exp_id)
            save_checkpoint(results, aggregated_results, list(completed_experiments))
            
            # Reset consecutive failures on success
            consecutive_failures = 0
            
        except Exception as e:
            error_msg = str(e)
            print(f"❌ Experiment failed: {error_msg}")
            
            # Check if this is a connection error
            if is_connection_error(error_msg):
                consecutive_failures += 1
                print(f"⚠️  Connection error detected ({consecutive_failures}/{CONSECUTIVE_FAILURE_LIMIT})")
                
                # Verify connection is actually down
                if not check_internet_connection():
                    print(f"\n🔌 CONNECTION LOST!")
                    print(f"   Exiting gracefully to preserve progress.")
                    print(f"   Successfully completed: {len(completed_experiments)} experiments")
                    print(f"   Remaining: {total_experiments - len(completed_experiments)} experiments")
                    print(f"\n   ➡️  Restore connection and re-run this cell to resume.")
                    graceful_exit = True
                    break
                
                if consecutive_failures >= CONSECUTIVE_FAILURE_LIMIT:
                    print(f"\n⚠️  {CONSECUTIVE_FAILURE_LIMIT} consecutive failures detected!")
                    print(f"   Pausing to check connection...")
                    
                    if not check_internet_connection():
                        print(f"\n🔌 CONNECTION LOST!")
                        print(f"   Exiting gracefully to preserve progress.")
                        graceful_exit = True
                        break
                    else:
                        print(f"   Connection seems OK, continuing...")
                        consecutive_failures = 0
            else:
                consecutive_failures = 0
            
            # Log the failure but do NOT mark as complete — will be retried on resume
            results.append({
                'learning_rate': lr, 'gamma': gamma,
                'net_dimension': net_dim, 'batch_size': bs,
                'window': window_idx + 1, 'error': error_msg,
                'exp_key': exp_key
            })
            save_checkpoint(results, aggregated_results, list(completed_experiments))

successful_count = len([r for r in results if 'error' not in r])
failed_count = len([r for r in results if 'error' in r])

if graceful_exit:
    print(f"\n{'='*70}")
    print(f"⏸️  HPO PAUSED DUE TO CONNECTION LOSS")
    print(f"{'='*70}")
    print(f"Progress saved: {len(completed_experiments)}/{total_experiments} experiments")
    print(f"\nTo resume: Restore internet connection and re-run this cell")
elif len(completed_experiments) < total_experiments:
    missing = total_experiments - len(completed_experiments)
    print(f"\n{'='*70}")
    print(f"⚠️  HPO run finished with {missing} experiments still incomplete!")
    print(f"{'='*70}")
    print(f"Completed: {len(completed_experiments)}/{total_experiments}")
    print(f"Successful: {successful_count} | Failed (will retry): {failed_count}")
    print(f"\n➡️  Re-run this cell to retry the {missing} remaining experiments")
else:
    print(f"\n{'='*70}")
    print(f"✅ Walk-forward grid search COMPLETE!")
    print(f"{'='*70}")
    print(f"All {total_experiments} experiments finished")
    print(f"Successful: {successful_count} | Failed: {failed_count}")

# Final save
save_checkpoint(results, aggregated_results, list(completed_experiments))

🔌 Checking connection to Alpaca API...
✅ Connection OK

📂 Checkpoint found from 2026-04-05 08:48:17
   Resuming from 243 completed experiments
✅ Resuming HPO - 243 successful experiments loaded

Testing 81 combinations across 3 windows
Total experiments: 243
Already completed: 243
Remaining: 0

✅ Walk-forward grid search COMPLETE!
All 243 experiments finished
Successful: 243 | Failed: 0
   💾 Checkpoint saved (243 experiments complete)


## Analyze Results

In [39]:
# Analyze Walk-Forward Results
# Filter by pre-committed HPO gates, rank by worst-case (min_sharpe)
from datetime import datetime

# ╔══════════════════════════════════════════════════════════════════════╗
# ║  PRE-COMMITTED HPO GATE — DO NOT MODIFY AFTER SEEING OOS RESULTS  ║
# ╚══════════════════════════════════════════════════════════════════════╝
HPO_GATE = {
    'min_sharpe_floor':    1.0,     # Worst-case Sharpe across all windows must exceed this
    'max_drawdown_floor': -0.10,    # Worst drawdown across all windows (negative, -10%)
    'min_mean_return':     0.0,     # Mean return across windows must be positive
}

TOP_K = 3  # Number of candidates to advance to OOS backtest
HPO_RANKING_METRIC = 'min_sharpe'  # Rank by worst-case, not peaks

HPO_GATE_TIMESTAMP = datetime.now().isoformat()
print(f'HPO Gate locked at: {HPO_GATE_TIMESTAMP}')
for k, v in HPO_GATE.items():
    print(f'  {k}: {v}')
print(f'Ranking metric: {HPO_RANKING_METRIC}')
print(f'Top-K candidates: {TOP_K}')

# --- Aggregate results ---
agg_summary = []
for exp_key, data in aggregated_results.items():
    if data['successful_windows'] == len(VALIDATION_WINDOWS):
        params = data['params']
        agg_summary.append({
            'exp_key': exp_key,
            'learning_rate': params['learning_rate'],
            'gamma': params['gamma'],
            'net_dimension': str(params['net_dimension']),
            'batch_size': params['batch_size'],
            'mean_sharpe': np.mean(data['sharpe_ratios']),
            'std_sharpe': np.std(data['sharpe_ratios'], ddof=1),
            'min_sharpe': np.min(data['sharpe_ratios']),
            'mean_return': np.mean(data['returns']),
            'worst_drawdown': np.min(data['max_drawdowns']),
            'sharpe_ratios': data['sharpe_ratios'],
            'returns': data['returns'],
            'max_drawdowns': data['max_drawdowns'],
        })

agg_df = pd.DataFrame(agg_summary)

if len(agg_df) > 0:
    print("\n" + "=" * 80)
    print("WALK-FORWARD AGGREGATED RESULTS")
    print(f"{len(agg_df)} configs succeeded on all {len(VALIDATION_WINDOWS)} windows")
    print("=" * 80)

    # --- Apply HPO gates ---
    pre_gate = len(agg_df)
    gated_df = agg_df[
        (agg_df['min_sharpe'] >= HPO_GATE['min_sharpe_floor']) &
        (agg_df['worst_drawdown'] >= HPO_GATE['max_drawdown_floor']) &
        (agg_df['mean_return'] >= HPO_GATE['min_mean_return'])
    ].copy()

    print(f"\nHPO Gate: {pre_gate} configs → {len(gated_df)} survivors")
    if len(gated_df) < pre_gate:
        rejected = pre_gate - len(gated_df)
        print(f"  Rejected {rejected} configs (min_sharpe < {HPO_GATE['min_sharpe_floor']}"
              f" or worst_dd < {HPO_GATE['max_drawdown_floor']:.0%}"
              f" or mean_return < {HPO_GATE['min_mean_return']})")

    if len(gated_df) > 0:
        # --- Rank survivors by worst-case Sharpe ---
        gated_df = gated_df.sort_values(HPO_RANKING_METRIC, ascending=False).reset_index(drop=True)

        # Display comparison tables
        print(f"\n{'='*80}")
        print(f"TOP {min(5, len(gated_df))} GATED CONFIGS BY {HPO_RANKING_METRIC.upper()}:")
        print("-" * 80)
        display_cols = ['exp_key', 'learning_rate', 'gamma', 'net_dimension', 'batch_size',
                        'min_sharpe', 'mean_sharpe', 'std_sharpe', 'mean_return', 'worst_drawdown']
        print(gated_df[display_cols].head().to_string())

        # Select top-K candidates
        candidates_df = gated_df.head(TOP_K).copy()
        print(f"\n{'='*80}")
        print(f"TOP-{TOP_K} CANDIDATES FOR OOS BACKTEST:")
        print("=" * 80)
        for rank, (_, row) in enumerate(candidates_df.iterrows(), 1):
            print(f"  #{rank}: {row['exp_key']}")
            print(f"       min_sharpe={row['min_sharpe']:.3f}  mean_sharpe={row['mean_sharpe']:.3f}"
                  f"  worst_dd={row['worst_drawdown']:.2%}  mean_ret={row['mean_return']:.2%}")

        # Save results
        results_df = pd.DataFrame([r for r in results if 'error' not in r])
        results_df.to_csv(f'./hpo_per_window_results{TIER_SUFFIX}.csv', index=False)
        agg_df.to_csv(f'./hpo_aggregated_results{TIER_SUFFIX}.csv', index=False)
        gated_df.to_csv(f'./hpo_gated_results{TIER_SUFFIX}.csv', index=False)
        print(f"\nResults saved to hpo_*_results{TIER_SUFFIX}.csv")
    else:
        print("\n⚠️ No configs passed HPO gates. Consider relaxing thresholds.")
        candidates_df = pd.DataFrame()
else:
    print("⚠️ No configurations succeeded on all validation windows.")
    candidates_df = pd.DataFrame()

HPO Gate locked at: 2026-04-05T09:36:51.902606
  min_sharpe_floor: 1.0
  max_drawdown_floor: -0.1
  min_mean_return: 0.0
Ranking metric: min_sharpe
Top-K candidates: 3

WALK-FORWARD AGGREGATED RESULTS
81 configs succeeded on all 3 windows

HPO Gate: 81 configs → 36 survivors
  Rejected 45 configs (min_sharpe < 1.0 or worst_dd < -10% or mean_return < 0.0)

TOP 5 GATED CONFIGS BY MIN_SHARPE:
--------------------------------------------------------------------------------
                          exp_key  learning_rate  gamma net_dimension  batch_size  min_sharpe  mean_sharpe  std_sharpe  mean_return  worst_drawdown
0    lr1e-05_g0.9_net64x32_bs1024        0.00001   0.90      [64, 32]        1024    5.334519     6.318745    1.195699     0.062443       -0.038331
1    lr3e-05_g0.9_net32x16_bs1024        0.00003   0.90      [32, 16]        1024    5.113315     5.522864    0.421371     0.058170       -0.040061
2    lr3e-05_g0.97_net64x32_bs256        0.00003   0.97      [64, 32]         256 

## Visualize Parameter Impact

In [41]:
import matplotlib.pyplot as plt

if len(agg_df) > 0:
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # 1. Learning rate vs Mean Sharpe
    lr_groups = agg_df.groupby('learning_rate')['mean_sharpe'].mean()
    axes[0, 0].bar(range(len(lr_groups)), lr_groups.values, color='steelblue')
    axes[0, 0].set_xticks(range(len(lr_groups)))
    axes[0, 0].set_xticklabels([f'{lr:.0e}' for lr in lr_groups.index])
    axes[0, 0].set_title('Learning Rate vs Mean Sharpe', fontsize=12, fontweight='bold')
    axes[0, 0].set_xlabel('Learning Rate')
    axes[0, 0].set_ylabel('Mean Sharpe Ratio')
    axes[0, 0].grid(True, alpha=0.3)
    
    # 2. Gamma vs Mean Sharpe
    gamma_groups = agg_df.groupby('gamma')['mean_sharpe'].mean()
    axes[0, 1].bar([str(g) for g in gamma_groups.index], gamma_groups.values, color='darkorange')
    axes[0, 1].set_title('Gamma vs Mean Sharpe', fontsize=12, fontweight='bold')
    axes[0, 1].set_xlabel('Gamma')
    axes[0, 1].set_ylabel('Mean Sharpe Ratio')
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3. Network size vs Mean Sharpe
    net_groups = agg_df.groupby('net_dimension')['mean_sharpe'].mean()
    axes[0, 2].bar(range(len(net_groups)), net_groups.values, color='forestgreen')
    axes[0, 2].set_xticks(range(len(net_groups)))
    axes[0, 2].set_xticklabels(net_groups.index, rotation=45)
    axes[0, 2].set_title('Network Size vs Mean Sharpe', fontsize=12, fontweight='bold')
    axes[0, 2].set_xlabel('Network Dimensions')
    axes[0, 2].set_ylabel('Mean Sharpe Ratio')
    axes[0, 2].grid(True, alpha=0.3)
    
    # 4. Batch size vs Mean Sharpe
    bs_groups = agg_df.groupby('batch_size')['mean_sharpe'].mean()
    axes[1, 0].bar([str(int(b)) for b in bs_groups.index], bs_groups.values, color='purple')
    axes[1, 0].set_title('Batch Size vs Mean Sharpe', fontsize=12, fontweight='bold')
    axes[1, 0].set_xlabel('Batch Size')
    axes[1, 0].set_ylabel('Mean Sharpe Ratio')
    axes[1, 0].grid(True, alpha=0.3)
    
    # 5. Min Sharpe vs Mean Sharpe scatter (worst-case vs average)
    axes[1, 1].scatter(agg_df['mean_sharpe'], agg_df['min_sharpe'], 
                       c=agg_df['worst_drawdown'], cmap='RdYlGn', s=100, alpha=0.7)
    axes[1, 1].set_title('Mean Sharpe vs Min Sharpe', fontsize=12, fontweight='bold')
    axes[1, 1].set_xlabel('Mean Sharpe Ratio')
    axes[1, 1].set_ylabel('Min Sharpe (Worst Window)')
    axes[1, 1].grid(True, alpha=0.3)
    cbar = plt.colorbar(axes[1, 1].collections[0], ax=axes[1, 1])
    cbar.set_label('Worst Drawdown')
    
    # 6. Per-window performance for top 5 configs
    results_df = pd.DataFrame([r for r in results if 'error' not in r])
    top5_keys = agg_df.sort_values('min_sharpe', ascending=False).head(5)['exp_key'].values
    for key in top5_keys:
        config_results = results_df[results_df['exp_key'] == key]
        axes[1, 2].plot(config_results['window'], config_results['sharpe_ratio'], 
                       marker='o', label=key[:25], linewidth=2)
    axes[1, 2].set_title('Top 5 Configs: Sharpe by Window', fontsize=12, fontweight='bold')
    axes[1, 2].set_xlabel('Validation Window')
    axes[1, 2].set_ylabel('Sharpe Ratio')
    axes[1, 2].legend(fontsize=8, loc='best')
    axes[1, 2].grid(True, alpha=0.3)
    axes[1, 2].set_xticks([1, 2, 3])
    
    plt.tight_layout()
    plt.savefig('./hpo_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Analysis plot saved to hpo_analysis.png")
else:
    print("No data to visualize.")

✅ Analysis plot saved to hpo_analysis.png


/tmp/ipykernel_1975790/3813857740.py:68: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# Gated Top-K OOS Validation

After filtering HPO results through **pre-committed gates** (min Sharpe floor, drawdown floor),
the top-K candidates are retrained and backtested on a **completely held-out period**
(Nov–Dec 2025). OOS gates then produce a binary **PASS/FAIL** verdict.

**Protocol**: HPO Grid → Pre-Committed Gate Filter → Rank by min_sharpe → Top-K Retrain → OOS Backtest → OOS Gate → VERDICT

Reference: Pham The Anh, *AlgoXpert Alpha Research Framework* — "select from stability regions, not peaks"

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  PRE-COMMITTED OOS GATE — DO NOT MODIFY AFTER SEEING OOS RESULTS  ║
# ╚══════════════════════════════════════════════════════════════════════╝

OOS_GATE = {
    # --- OOS backtest performance ---
    'min_sharpe':         0.5,      # Annualised Sharpe on held-out period
    'max_drawdown':      -0.15,     # Max DD floor (negative, -15%)
    'min_return':         0.0,      # Must be profitable on OOS

    # --- Degradation bounds (HPO → OOS) ---
    'min_sharpe_retention': 0.3,    # OOS Sharpe / HPO mean Sharpe must be > 30%
    'dd_integrity_margin':  1.5,    # OOS MaxDD must be < 1.5× HPO worst DD

    # --- Actions ---
    'pass_action':  'DEPLOY — parameters locked, copy to v2 notebook',
    'fail_action':  'REJECT — no rescue optimization, re-run HPO with new ranges',
}

OOS_GATE_TIMESTAMP = datetime.now().isoformat()
print(f'OOS Gate locked at: {OOS_GATE_TIMESTAMP}')
for k, v in OOS_GATE.items():
    if not k.endswith('_action'):
        print(f'  {k}: {v}')

# Training period: Feb-Oct 2025 (covers all HPO windows)
BACKTEST_TRAIN_START = '2025-02-01'
BACKTEST_TRAIN_END = '2025-10-31'

# Test period: Nov-Dec 2025 (completely held-out from HPO)
BACKTEST_TEST_START = '2025-11-01'
BACKTEST_TEST_END = '2025-12-31'

# Build candidate param dicts
CANDIDATE_PARAMS = []
if len(candidates_df) > 0:
    for rank, (_, row) in enumerate(candidates_df.iterrows(), 1):
        params = {
            "learning_rate": float(row['learning_rate']),
            "batch_size": int(row['batch_size']),
            "gamma": float(row['gamma']),
            "seed": 312,
            "net_dimension": eval(row['net_dimension']),
            "target_step": 5000,
            "eval_gap": 30,
            "eval_times": 1,
        }
        CANDIDATE_PARAMS.append({
            'rank': rank,
            'exp_key': row['exp_key'],
            'erl_params': params,
            'hpo_min_sharpe': row['min_sharpe'],
            'hpo_mean_sharpe': row['mean_sharpe'],
            'hpo_worst_dd': row['worst_drawdown'],
            'hpo_mean_return': row['mean_return'],
        })

    print(f"\n{len(CANDIDATE_PARAMS)} candidates ready for OOS backtest:")
    print(f"Training: {BACKTEST_TRAIN_START} to {BACKTEST_TRAIN_END}")
    print(f"OOS Test: {BACKTEST_TEST_START} to {BACKTEST_TEST_END} (held-out)")
    for c in CANDIDATE_PARAMS:
        print(f"  #{c['rank']}: {c['exp_key']} (HPO min_sharpe={c['hpo_min_sharpe']:.3f})")

    # Pre-cache OOS data (train + test downloaded once, shared across all candidates)
    print(f"\n📥 Pre-caching OOS data...")
    download_and_cache(BACKTEST_TRAIN_START, BACKTEST_TRAIN_END, ticker_list,
                       'alpaca', '1Min', INDICATORS, True,
                       API_KEY=API_KEY, API_SECRET=API_SECRET,
                       API_BASE_URL=API_BASE_URL)
    download_and_cache(BACKTEST_TEST_START, BACKTEST_TEST_END, ticker_list,
                       'alpaca', '1Min', INDICATORS, True,
                       API_KEY=API_KEY, API_SECRET=API_SECRET,
                       API_BASE_URL=API_BASE_URL)
    print(f"✅ OOS data cached")
else:
    print("⚠️ No candidates to backtest.")

OOS Gate locked at: 2026-04-05T09:40:54.647680
  min_sharpe: 0.5
  max_drawdown: -0.15
  min_return: 0.0
  min_sharpe_retention: 0.3
  dd_integrity_margin: 1.5

3 candidates ready for OOS backtest:
Training: 2025-02-01 to 2025-10-31
OOS Test: 2025-11-01 to 2025-12-31 (held-out)
  #1: lr1e-05_g0.9_net64x32_bs1024 (HPO min_sharpe=5.335)
  #2: lr3e-05_g0.9_net32x16_bs1024 (HPO min_sharpe=5.113)
  #3: lr3e-05_g0.97_net64x32_bs256 (HPO min_sharpe=4.799)


In [ ]:
# === TRAIN TOP-K CANDIDATES ===
oos_train_arrays = _data_cache.get((BACKTEST_TRAIN_START, BACKTEST_TRAIN_END))

for candidate in CANDIDATE_PARAMS:
    rank = candidate['rank']
    exp_key = candidate['exp_key']
    cwd = f'./backtest_candidate_{rank}{TIER_SUFFIX}'
    candidate['cwd'] = cwd

    print(f"\n{'='*70}")
    print(f"Training candidate #{rank}: {exp_key}")
    print(f"Period: {BACKTEST_TRAIN_START} to {BACKTEST_TRAIN_END}")
    print(f"Params: lr={candidate['erl_params']['learning_rate']:.0e}"
          f"  gamma={candidate['erl_params']['gamma']}"
          f"  net={candidate['erl_params']['net_dimension']}"
          f"  bs={candidate['erl_params']['batch_size']}")
    print(f"{'='*70}")

    train(
        start_date=BACKTEST_TRAIN_START,
        end_date=BACKTEST_TRAIN_END,
        ticker_list=ticker_list,
        data_source='alpaca',
        time_interval='1Min',
        technical_indicator_list=INDICATORS,
        drl_lib='elegantrl',
        env=env,
        model_name='ppo',
        if_vix=True,
        API_KEY=API_KEY,
        API_SECRET=API_SECRET,
        API_BASE_URL=API_BASE_URL,
        erl_params=candidate['erl_params'],
        cwd=cwd,
        break_step=2e5,
        account_config=ACCOUNT_CONFIG,
        cached_arrays=oos_train_arrays,
    )

    print(f"  ✅ #{rank} training complete → {cwd}")print(f"\nAll {len(CANDIDATE_PARAMS)} candidates trained.")



Training candidate #1: lr1e-05_g0.9_net64x32_bs1024
Period: 2025-02-01 to 2025-10-31
Params: lr=1e-05  gamma=0.9  net=[64, 32]  bs=1024
Alpaca successfully connected
Data cleaning started
align start and end dates
produce full timestamp index
Start processing tickers
The price of the first row for ticker AAPL is NaN. It will be filled with the first valid price.
The price of the first row for ticker AMGN is NaN. It will be filled with the first valid price.
The price of the first row for ticker AXP is NaN. It will be filled with the first valid price.
The price of the first row for ticker BA is NaN. It will be filled with the first valid price.
The price of the first row for ticker CAT is NaN. It will be filled with the first valid price.
The price of the first row for ticker CRM is NaN. It will be filled with the first valid price.
The price of the first row for ticker CSCO is NaN. It will be filled with the first valid price.
The price of the first row for ticker CVX is NaN. It will

In [53]:
# === BACKTEST TOP-K CANDIDATES ON HELD-OUT PERIOD ===
oos_test_arrays = _data_cache.get((BACKTEST_TEST_START, BACKTEST_TEST_END))

for candidate in CANDIDATE_PARAMS:
    rank = candidate['rank']
    exp_key = candidate['exp_key']
    cwd = candidate['cwd']

    print(f"\nBacktesting candidate #{rank}: {exp_key}")
    print(f"OOS Period: {BACKTEST_TEST_START} to {BACKTEST_TEST_END}")

    account_values = test(
        start_date=BACKTEST_TEST_START,
        end_date=BACKTEST_TEST_END,
        ticker_list=ticker_list,
        data_source='alpaca',
        time_interval='1Min',
        technical_indicator_list=INDICATORS,
        drl_lib='elegantrl',
        env=env,
        model_name='ppo',
        if_vix=True,
        API_KEY=API_KEY,
        API_SECRET=API_SECRET,
        API_BASE_URL=API_BASE_URL,
        cwd=cwd,
        net_dimension=candidate['erl_params']['net_dimension'],
        account_config=ACCOUNT_CONFIG,
        cached_arrays=oos_test_arrays,
    )

    metrics = calculate_metrics(account_values)
    candidate['oos_account_values'] = account_values
    candidate['oos_metrics'] = metrics

    print(f"  Sharpe={metrics['sharpe_ratio']:.3f}  Return={metrics['total_return']:.2%}"
          f"  MaxDD={metrics['max_drawdown']:.2%}  Calmar={metrics['calmar_ratio']:.3f}")

# Summary table
print(f"\n{'='*80}")
print(f"OOS BACKTEST RESULTS — ALL CANDIDATES")
print(f"{'='*80}")
print(f"{'#':<4} {'Config':<40} {'Sharpe':>8} {'Return':>8} {'MaxDD':>8} {'Calmar':>8}")
print("-" * 80)
for c in CANDIDATE_PARAMS:
    m = c['oos_metrics']
    print(f"#{c['rank']:<3} {c['exp_key']:<40} {m['sharpe_ratio']:>8.3f} {m['total_return']:>7.2%}"
          f" {m['max_drawdown']:>7.2%} {m['calmar_ratio']:>8.3f}")


Backtesting candidate #1: lr1e-05_g0.9_net64x32_bs1024
OOS Period: 2025-11-01 to 2025-12-31
  [downloading] 2025-11-01 → 2025-12-31 ...
Alpaca successfully connected
Data cleaning started
align start and end dates
produce full timestamp index
Start processing tickers
ticker list complete
Start concat and rename
Data clean finished!
Started adding Indicators
Running Loop
Restore Timestamps
Finished adding Indicators
Data cleaning started
align start and end dates
produce full timestamp index
Start processing tickers
ticker list complete
Start concat and rename
Data clean finished!
price_array:  15990
| load actor from: ./backtest_candidate_1_1m/actor.pth


/tmp/ipykernel_1975790/383902926.py:108: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  s_tensor = _torch.as_tensor((state,), device=act_device)


Test Finished!
episode_return 0.9956166386320746
  Sharpe=0.134  Return=-0.44%  MaxDD=-5.08%  Calmar=-0.525

Backtesting candidate #2: lr3e-05_g0.9_net32x16_bs1024
OOS Period: 2025-11-01 to 2025-12-31
  [cache hit] 2025-11-01 → 2025-12-31
price_array:  15990
| load actor from: ./backtest_candidate_2_1m/actor.pth
Test Finished!
episode_return 1.0386942349719164
  Sharpe=1.933  Return=3.87%  MaxDD=-5.52%  Calmar=4.761

Backtesting candidate #3: lr3e-05_g0.97_net64x32_bs256
OOS Period: 2025-11-01 to 2025-12-31
  [cache hit] 2025-11-01 → 2025-12-31
price_array:  15990
| load actor from: ./backtest_candidate_3_1m/actor.pth
Test Finished!
episode_return 1.0258814655301114
  Sharpe=1.390  Return=2.59%  MaxDD=-4.00%  Calmar=4.256

OOS BACKTEST RESULTS — ALL CANDIDATES
#    Config                                     Sharpe   Return    MaxDD   Calmar
--------------------------------------------------------------------------------
#1   lr1e-05_g0.9_net64x32_bs1024                0.134  -0.44%  -5

In [54]:
# === VISUALIZE ALL CANDIDATES ===
n_candidates = len(CANDIDATE_PARAMS)
fig, axes = plt.subplots(n_candidates, 2, figsize=(14, 5 * n_candidates))
if n_candidates == 1:
    axes = axes.reshape(1, -1)

colors = ['blue', 'green', 'orange', 'purple', 'brown']

for idx, candidate in enumerate(CANDIDATE_PARAMS):
    av = candidate['oos_account_values']
    m = candidate['oos_metrics']
    color = colors[idx % len(colors)]
    label = f"#{candidate['rank']}: {candidate['exp_key']}"

    # Portfolio Value
    ax = axes[idx, 0]
    ax.plot(av, linewidth=2, color=color)
    ax.axhline(y=av[0], color='gray', linestyle='--', alpha=0.7)
    ax.fill_between(range(len(av)), av[0], av,
                    where=[v > av[0] for v in av], alpha=0.2, color='green')
    ax.fill_between(range(len(av)), av[0], av,
                    where=[v < av[0] for v in av], alpha=0.2, color='red')
    ax.set_title(f'{label}\nSharpe={m["sharpe_ratio"]:.2f} Return={m["total_return"]:.1%} DD={m["max_drawdown"]:.1%}',
                 fontsize=11, fontweight='bold')
    ax.set_ylabel('Portfolio Value ($)')
    ax.grid(True, alpha=0.3)

    # Drawdown
    ax = axes[idx, 1]
    running_max = np.maximum.accumulate(av)
    drawdowns = (np.array(av) - running_max) / running_max * 100
    ax.fill_between(range(len(drawdowns)), drawdowns, 0, alpha=0.7, color='red')
    ax.axhline(y=m['max_drawdown']*100, color='darkred', linestyle='--',
               linewidth=2, label=f"Max DD: {m['max_drawdown']:.1%}")
    ax.set_title(f'Drawdown — {label}', fontsize=11, fontweight='bold')
    ax.set_ylabel('Drawdown (%)')
    ax.legend()
    ax.grid(True, alpha=0.3)

axes[-1, 0].set_xlabel('Time Steps')
axes[-1, 1].set_xlabel('Time Steps')
plt.tight_layout()
plt.savefig(f'./backtest_candidates{TIER_SUFFIX}.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to backtest_candidates{TIER_SUFFIX}.png")

Plot saved to backtest_candidates_1m.png


/tmp/ipykernel_1975790/4240475630.py:44: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
# === OOS GATE EVALUATION + FINAL VERDICT ===
import json as _json

print("=" * 70)
print("           OOS DEPLOYMENT GATE — EVALUATION")
print("=" * 70)
print(f"Gate locked:    {OOS_GATE_TIMESTAMP}")
print(f"Evaluated:      {datetime.now().isoformat()}")
print()

best_candidate = None

for candidate in CANDIDATE_PARAMS:
    m = candidate['oos_metrics']
    rank = candidate['rank']
    exp_key = candidate['exp_key']

    verdicts = {}

    # --- G1: OOS Sharpe ---
    verdicts['G1_OOS_Sharpe'] = (
        m['sharpe_ratio'] >= OOS_GATE['min_sharpe'],
        f"{m['sharpe_ratio']:.3f}",
        f">= {OOS_GATE['min_sharpe']}",
    )

    # --- G2: OOS Max Drawdown ---
    verdicts['G2_OOS_MaxDD'] = (
        m['max_drawdown'] >= OOS_GATE['max_drawdown'],
        f"{m['max_drawdown']:.2%}",
        f">= {OOS_GATE['max_drawdown']:.0%}",
    )

    # --- G3: OOS Return ---
    verdicts['G3_OOS_Return'] = (
        m['total_return'] >= OOS_GATE['min_return'],
        f"{m['total_return']:.2%}",
        f">= {OOS_GATE['min_return']:.0%}",
    )

    # --- G4: Sharpe degradation (HPO → OOS) ---
    hpo_mean = candidate['hpo_mean_sharpe']
    retention = m['sharpe_ratio'] / hpo_mean if hpo_mean > 0 else 0
    verdicts['G4_Degradation'] = (
        retention >= OOS_GATE['min_sharpe_retention'],
        f"{retention:.2f} ({m['sharpe_ratio']:.2f}/{hpo_mean:.2f})",
        f">= {OOS_GATE['min_sharpe_retention']}",
    )

    # --- G5: Drawdown integrity ---
    hpo_worst_dd = candidate['hpo_worst_dd']
    dd_limit = hpo_worst_dd * OOS_GATE['dd_integrity_margin']
    dd_ok = m['max_drawdown'] >= dd_limit
    verdicts['G5_DD_Integrity'] = (
        dd_ok,
        f"{m['max_drawdown']:.2%} vs limit {dd_limit:.2%}",
        f"OOS DD > {OOS_GATE['dd_integrity_margin']}× HPO worst",
    )

    all_pass = all(v[0] for v in verdicts.values())
    candidate['verdicts'] = verdicts
    candidate['all_pass'] = all_pass
    candidate['sharpe_retention'] = retention

    status = '\033[92mPASS\033[0m' if all_pass else '\033[91mFAIL\033[0m'
    print(f"--- Candidate #{rank}: {exp_key} → {status} ---")
    for gate, (passed, val, thresh) in verdicts.items():
        glyph = 'PASS' if passed else '*** FAIL ***'
        print(f"  {gate:<22} {glyph:<14} {val:<30} {thresh}")
    print()

# --- Select winner among passing candidates ---
passing = [c for c in CANDIDATE_PARAMS if c['all_pass']]

print("=" * 70)
if passing:
    # Pick best OOS Sharpe among passers (pre-committed OOS ranking)
    winner = max(passing, key=lambda c: c['oos_metrics']['sharpe_ratio'])
    best_candidate = winner

    BEST_PARAMS = winner['erl_params']
    backtest_cwd = winner['cwd']
    backtest_metrics = winner['oos_metrics']
    backtest_account_values = winner['oos_account_values']

    print(f"\n  >>> VERDICT: \033[1;92mDEPLOY\033[0m <<<")
    print(f"  {OOS_GATE['pass_action']}")
    print(f"\n  Winner: #{winner['rank']} {winner['exp_key']}")
    print(f"  OOS Sharpe={backtest_metrics['sharpe_ratio']:.3f}"
          f"  Return={backtest_metrics['total_return']:.2%}"
          f"  MaxDD={backtest_metrics['max_drawdown']:.2%}"
          f"  Calmar={backtest_metrics['calmar_ratio']:.3f}")
    print(f"  Sharpe retention (HPO→OOS): {winner['sharpe_retention']:.0%}")

    # Degradation diagnostics (informational)
    print(f"\n  --- Degradation Diagnostics ---")
    print(f"  HPO min_sharpe: {winner['hpo_min_sharpe']:.3f}")
    print(f"  HPO mean_sharpe: {winner['hpo_mean_sharpe']:.3f}")
    print(f"  OOS Sharpe:      {backtest_metrics['sharpe_ratio']:.3f}")
    print(f"  HPO worst DD:    {winner['hpo_worst_dd']:.2%}")
    print(f"  OOS MaxDD:       {backtest_metrics['max_drawdown']:.2%}")

    print(f"\n{'='*70}")
    print("FINAL RECOMMENDED PARAMS FOR v2 NOTEBOOK:")
    print("=" * 70)
    print(f'''
ERL_PARAMS = {{
    "learning_rate": {BEST_PARAMS['learning_rate']:.0e},
    "batch_size": {BEST_PARAMS['batch_size']},
    "gamma": {BEST_PARAMS['gamma']},
    "seed": 312,
    "net_dimension": {BEST_PARAMS['net_dimension']},
    "target_step": 5000,
    "eval_gap": 30,
    "eval_times": 1
}}
''')
else:
    print(f"\n  >>> VERDICT: \033[1;91mREJECT\033[0m <<<")
    print(f"  {OOS_GATE['fail_action']}")
    print(f"  No candidates passed all OOS gates.")
    for c in CANDIDATE_PARAMS:
        failed = [g for g, (p, _, _) in c['verdicts'].items() if not p]
        print(f"  #{c['rank']} {c['exp_key']}: failed {', '.join(failed)}")

print("=" * 70)

# --- Evidence Ledger ---
ledger_dir = './gate_ledger'
os.makedirs(ledger_dir, exist_ok=True)

record = {
    'hpo_gate_timestamp': HPO_GATE_TIMESTAMP,
    'oos_gate_timestamp': OOS_GATE_TIMESTAMP,
    'eval_timestamp': datetime.now().isoformat(),
    'hpo_gate': HPO_GATE,
    'oos_gate': {k: v for k, v in OOS_GATE.items() if not k.endswith('_action')},
    'hpo_ranking_metric': HPO_RANKING_METRIC,
    'top_k': TOP_K,
    'candidates': [
        {
            'rank': c['rank'],
            'exp_key': c['exp_key'],
            'hpo_min_sharpe': c['hpo_min_sharpe'],
            'hpo_mean_sharpe': c['hpo_mean_sharpe'],
            'hpo_worst_dd': c['hpo_worst_dd'],
            'oos_sharpe': c['oos_metrics']['sharpe_ratio'],
            'oos_return': c['oos_metrics']['total_return'],
            'oos_max_dd': c['oos_metrics']['max_drawdown'],
            'oos_calmar': c['oos_metrics']['calmar_ratio'],
            'gates_passed': c['all_pass'],
            'verdicts': {k: {'passed': v[0], 'value': v[1], 'threshold': v[2]}
                         for k, v in c['verdicts'].items()},
        }
        for c in CANDIDATE_PARAMS
    ],
    'final_verdict': 'DEPLOY' if passing else 'REJECT',
    'winner': best_candidate['exp_key'] if best_candidate else None,
}

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
ledger_path = os.path.join(ledger_dir, f'gate_{ts}.json')
with open(ledger_path, 'w') as f:
    _json.dump(record, f, indent=2, default=str)

print(f"\nEvidence logged: {ledger_path}")

           OOS DEPLOYMENT GATE — EVALUATION
Gate locked:    2026-04-05T09:40:54.647680
Evaluated:      2026-04-05T10:35:00.673060

--- Candidate #1: lr1e-05_g0.9_net64x32_bs1024 → FAIL ---
  G1_OOS_Sharpe          *** FAIL ***   0.134                          >= 0.5
  G2_OOS_MaxDD           PASS           -5.08%                         >= -15%
  G3_OOS_Return          *** FAIL ***   -0.44%                         >= 0%
  G4_Degradation         *** FAIL ***   0.02 (0.13/6.32)               >= 0.3
  G5_DD_Integrity        PASS           -5.08% vs limit -5.75%         OOS DD > 1.5× HPO worst

--- Candidate #2: lr3e-05_g0.9_net32x16_bs1024 → PASS ---
  G1_OOS_Sharpe          PASS           1.933                          >= 0.5
  G2_OOS_MaxDD           PASS           -5.52%                         >= -15%
  G3_OOS_Return          PASS           3.87%                          >= 0%
  G4_Degradation         PASS           0.35 (1.93/5.52)               >= 0.3
  G5_DD_Integrity        PASS    

: 